In [1]:
import pandas as pd

Import Dataset:

In [2]:
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"

df = pd.read_csv(url)

df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


Data preparation

In [3]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [4]:
categorical = ['lead_source', 'industry', 'employment_status', 'location']
numerical = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']

In [5]:
df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

1. Mode of industry

In [6]:
df['industry'].value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

In [7]:
df['industry'].mode()[0]

'technology'

2. Correlation matrix

In [8]:
corr = df[numerical].corr()

corr

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [9]:
pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count'),
]

for a, b in pairs:
    print(f"{a} and {b}: {corr.loc[a, b]:.4f}")

interaction_count and lead_score: 0.9157
number_of_courses_viewed and lead_score: 0.7572
number_of_courses_viewed and interaction_count: 0.7216
annual_income and interaction_count: 0.1228


Split the data

In [10]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

In [11]:
y_train = df_train['converted'].values
y_val = df_val['converted'].values

df_train = df_train.drop(columns=['converted']).reset_index(drop=True)
df_val = df_val.drop(columns=['converted']).reset_index(drop=True)

len(df_train), len(df_val), len(df_test)

(3000, 1000, 1000)

3. Mutual information

In [12]:
from sklearn.metrics import mutual_info_score

for c in categorical:
    score = mutual_info_score(df_train[c], y_train)
    print(f"{c}: {round(score, 2)}")

lead_source: 0.03
industry: 0.0
employment_status: 0.02
location: 0.0


4. Logistic regression

In [13]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

features = categorical + numerical

dv = DictVectorizer(sparse=False)

X_train = dv.fit_transform(df_train[features].to_dict(orient='records'))
X_val = dv.transform(df_val[features].to_dict(orient='records'))

In [14]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)

accuracy = accuracy_score(y_val, y_pred)

accuracy, round(accuracy, 2)

(0.645, 0.65)

5. Feature elimination

In [15]:
def train_and_score(feats, C=1.0):
    dv = DictVectorizer(sparse=False)
    X_tr = dv.fit_transform(df_train[feats].to_dict(orient='records'))
    X_va = dv.transform(df_val[feats].to_dict(orient='records'))

    m = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    m.fit(X_tr, y_train)

    return accuracy_score(y_val, m.predict(X_va))

In [16]:
for f in ['lead_source', 'number_of_courses_viewed', 'interaction_count']:
    acc = train_and_score([x for x in features if x != f])
    print(f"without {f}: accuracy={acc:.4f}, difference={accuracy - acc:.4f}")

without lead_source: accuracy=0.6420, difference=0.0030
without number_of_courses_viewed: accuracy=0.6430, difference=0.0020
without interaction_count: accuracy=0.6010, difference=0.0440


6. Regularized logistic regression

In [17]:
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    acc = train_and_score(features, C=C)
    print(f"C={C}: accuracy={round(acc, 3)}")

C=1e-06: accuracy=0.598
C=1e-05: accuracy=0.598
C=0.0001: accuracy=0.613


C=0.001: accuracy=0.645
